# Train 3D CNN Particle Classifier
This notebook builds a balanced particle/non-particle training dataset from detector proposals, then trains a 3D CNN classifier with PyTorch Lightning.

## Release Notes
- Configure one class workflow at a time (ribosome or hsp60).
- Stage-1 candidate CSVs are written to `./temp/*.csv`.
- Set crop/radius hyperparameters to match the chosen class.
- Update train/val split when using multiple tomograms.

In [ ]:
import importlib
import os
import sys

import numpy as np
import pandas as pd
import pytorch_lightning as pl
import torch
from pytorch_lightning.callbacks import ModelCheckpoint
from torch.utils.data import DataLoader

sys.path.append("../src")

import data
import modules
import postprocess
import utils

os.makedirs("./temp", exist_ok=True)

/data/biosoftware/miniconda3/miniconda3/envs/tomognn/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


<module 'data' from '/home/feity/cryoem/notebooks/../src/data.py'>

## Environment and Imports
This section loads Python dependencies, project modules, and creates `./temp` for intermediate CSV artifacts.
Run this once at the start of a fresh kernel.

In [3]:
# tomograms for ribosome
# tomo_paths = {
#     "test1": "/data/transformer_project/transforemer_model/release_models/data_example/ribosome.mrc"
# }
# tomograms for hsp60
tomo_paths = {
    "test2": "/data/transformer_project/transforemer_model/release_models/data_example/hsp60.mrc"
}

## Select Input Tomograms
Define the tomogram set used for training data generation.
Keep one class setup active at a time (ribosome or hsp60) so labels and checkpoints stay aligned.

In [ ]:
# Stage 1 detector (choose one model block).
import torch

# ribosome
# model = utils.loadModel("/data/transformer_project/transforemer_model/release_models/ribosome", "last.ckpt")
# model = model.eval()
# hsp60
model = utils.loadModel("/data/transformer_project/transforemer_model/release_models/hsp60", "last.ckpt")
model = model.eval()

# Generate slice-wise detector proposals and save to CSV.
if torch.cuda.is_available():
    model = model.cuda(0)

for i in tomo_paths:
    dataset = data.TestDatasetMrc(
        tomo_paths[i],
        norm="hist",
        reshape=800,
        length_for_average=3,
        gap=1,
    )
    df = postprocess.generatedfBySlice(model, dataset, gap=1, columns=["ribosome", "None"] )
    df.to_csv(f"./temp/{i}.csv", index=False)

model at stage  stage 1
model with output classes 2
model receiving class weights tensor([1.0000, 0.2000])
using consistency regularization coef 0.5


/data/biosoftware/miniconda3/miniconda3/envs/tomognn/lib/python3.11/site-packages/torch/nn/modules/transformer.py:307: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(f"enable_nested_tensor is True, but self.use_nested_tensor is False because {why_not_sparsity_fast_path}")
/home/feity/cryoem/notebooks/../src/utils.py:1397: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will

incompatible parameters []
finish loading parameters
loading test dataset


100%|██████████| 500/500 [01:28<00:00,  5.63it/s]


finish stage1


Stage 2 slices: 100%|██████████| 500/500 [00:29<00:00, 16.85it/s]


## Stage-1 Detector Inference
Run the stage-1 detector on each tomogram and save slice-wise predictions to `./temp/*.csv`.
These proposal CSV files are used to build positive/negative training samples for the 3D CNN.

In [ ]:
# Build a balanced training dataframe from detector proposals + GT labels.
# For multiple tomograms, process each one and concatenate all rows.
PROB_THRES = 0.20
TOMOGRAM_SIZE_X = 1024
TOMOGRAM_SIZE_Y = 1024
TOMOGRAM_SIZE_Z = 500
NEGATIVE_DIS = 35.0
# ribosome
# SWEEP_THRES = 15
# hsp60
SWEEP_THRES = 10

# Set this to the detector class column you are using (e.g., "ribosome" or "hsp60").
SCORE_COLUMN = "ribosome"

# Ground-truth labels (choose one block).
# labels = np.loadtxt("/data/transformer_project/transforemer_model/release_models/data_example/ribosome_label.txt")
# labels = pd.DataFrame(labels, columns=["z", "y", "x"] )
# labels["tomogram"] = "test1"

labels = np.loadtxt("/data/transformer_project/transforemer_model/release_models/data_example/hsp60_label.txt")
labels = pd.DataFrame(labels, columns=["z", "y", "x"] )
labels["tomogram"] = "test2"

dfs = []
for i in tomo_paths:
    df_predicts = utils.sweep_to_find_prediction_centers(
        tomogram_name=i,
        subdf_csv_path=f"./temp/{i}.csv",
        prob_threshold=PROB_THRES,
        tomogram_size_x=TOMOGRAM_SIZE_X,
        tomogram_size_y=TOMOGRAM_SIZE_Y,
        tomogram_size_z=TOMOGRAM_SIZE_Z,
        sweep_threshold=SWEEP_THRES,
        score_column=SCORE_COLUMN,
    )
    predict_center = df_predicts[["z", "y", "x"]].to_numpy()
    sub_label = labels[labels["tomogram"] == i][["z", "y", "x"]].to_numpy()

    neg_points, closest_dist = utils.build_balanced_negative_points_zyx(
        labels_zyx=sub_label,
        predict_center_zyx=predict_center,
        negative_distance=NEGATIVE_DIS,
        shape_zyx=(TOMOGRAM_SIZE_Z, TOMOGRAM_SIZE_Y, TOMOGRAM_SIZE_X),
        negative_to_positive_ratio=2,
        seed=42,
    )
    print(
        f"balance {i}: pos={len(sub_label)}, neg={len(neg_points)}, "
        f"ratio={len(neg_points) / max(len(sub_label), 1):.2f}"
    )

    k = [1] * len(sub_label) + [0] * len(neg_points)
    values = np.concatenate([sub_label, neg_points], axis=0)
    df = pd.DataFrame(values, columns=["z", "y", "x"] )
    df["label"] = k
    df["tomogram"] = i
    dfs.append(df)

df = pd.concat(dfs, axis=0)

balance test1: pos=321, neg=642, ratio=2.00


## Build Balanced Training Samples
Convert detector proposals into candidate centers and combine with ground truth to construct positive/negative samples.
Negative samples are generated with distance constraints to reduce near-positive ambiguity.

In [6]:
train = list(tomo_paths.keys())
val = list(tomo_paths.keys())

tomo_paths_train = {k: tomo_paths[k] for k in train}
tomo_paths_val = {k: tomo_paths[k] for k in val} 

train_df = df[df["tomogram"].isin(tomo_paths_train.keys())].reset_index(drop=True)
val_df = df[df["tomogram"].isin(tomo_paths_val.keys())].reset_index(drop=True)


## Train/Validation Split
Select tomograms for train and validation and materialize `train_df`/`val_df`.
For realistic evaluation, use disjoint tomogram sets when multiple tomograms are available.

In [ ]:
# Choose crop size and center radius to match target class.
# ribosome
# CROP_SIZE = 65
# R = 15
# hsp60
CROP_SIZE = 41
R = 12

train_dataset = data.Particle3DDataset(
    train_df,
    tomo_paths_train,
    crop_size=CROP_SIZE,
    norm="hist",
    r=R,
    if_augmentation=True,
    output_center=True,
 )

val_dataset = data.Particle3DDataset(
    val_df,
    tomo_paths_val,
    crop_size=CROP_SIZE,
    norm="hist",
    r=R,
    if_augmentation=True,
    output_center=True,
 )

## Dataset Construction
Create `Particle3DDataset` objects with class-specific crop size and optional center jitter augmentation.
This prepares volumetric crops for binary classification and auxiliary center supervision.

In [8]:
importlib.reload(data)
train_loader = DataLoader(
    train_dataset, batch_size=32, shuffle=True, num_workers=4, collate_fn=data.collate_with_aux
  )
val_loader = DataLoader(
    val_dataset, batch_size=32, shuffle=False, num_workers=4, collate_fn=data.collate_with_aux
  )
batch = next(iter(train_loader))
print("batch shapes:", batch[0].shape, batch[1].shape)


batch shapes: torch.Size([32, 1, 41, 41, 41]) torch.Size([32])


## DataLoader Sanity Check
Build train/validation dataloaders and inspect one batch to confirm tensor shapes and collate behavior.
If shapes are unexpected, verify crop size, augmentation flags, and collate function.

In [ ]:
out_dir = "."

# hsp60
model = modules.ParticleID3DNet_Binary(center_aux_enabled=True, small=True)

# ribosome
# model = modules.ParticleID3DNet_Binary(center_aux_enabled=True)

checkpoint_callback = ModelCheckpoint(
    dirpath=out_dir + f"/test_{CROP_SIZE}",
    filename="best",
    monitor="val_aupr",
    mode="max",
    save_top_k=1,
    save_last=True,
 )

checkpoint_callback2 = ModelCheckpoint(
    every_n_epochs=10,
    save_top_k=-1,
    dirpath=out_dir + f"/test_{CROP_SIZE}",
    filename="epoch-{epoch}",
    save_last=False,
 )

# remember to change to GPU if GPU is available.
trainer = pl.Trainer(
    max_epochs=20,
    accelerator="auto",
    devices=1,
    callbacks=[checkpoint_callback, checkpoint_callback2],
    log_every_n_steps=10,
 )

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


## Model and Trainer Configuration
Initialize the 3D CNN model, checkpoint callbacks, and Lightning trainer settings.
Adjust output directory and callbacks here before launching training.

## Start Training
Run model fitting on the configured train/validation dataloaders.
Training logs and checkpoints are saved to the configured output path.

In [10]:
trainer.fit(model, train_loader, val_loader)

You are using a CUDA device ('NVIDIA GeForce RTX 4090') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3]

   | Name             | Type              | Params | Mode 
----------------------------------------------------------------
0  | conv1            | Conv3d            | 2.0 K  | train
1  | bn1              | BatchNorm3d       | 32     | train
2  | conv1_1          | Conv3d            | 32.0 K | train
3  | conv2            | Conv3d            | 64.0 K | train
4  | bn2              | BatchNorm3d       | 64     | train
5  | conv2_1          | Conv3d            | 128 K  | train
6  | conv3            | Conv3d            | 256 K  | train
7  | bn3              | BatchNorm3d       | 128   

Sanity Checking DataLoader 0: 100%|██████████| 2/2 [00:00<00:00,  6.68it/s]val_aupr 1.0 val_auroc nan


/data/biosoftware/miniconda3/miniconda3/envs/tomognn/lib/python3.11/site-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


Epoch 19: 100%|██████████| 31/31 [00:02<00:00, 11.80it/s, v_num=38, val_center_loss=1.260]

`Trainer.fit` stopped: `max_epochs=20` reached.


Epoch 19: 100%|██████████| 31/31 [00:02<00:00, 10.73it/s, v_num=38, val_center_loss=1.260]
